# E11 · 课程学习实战：自适应池化 10×10 迁移至 20×20 大棋盘（Colab L4 GPU）

> **实验背景与核心目标**：
> 1. **历史对照**：在 E7 实验中，直接在 20×20 棋盘上从零随机初始化训练全棋盘 CNN，耗时 45 分钟（48 万步）**最终得分 0.00（100% 超时绕圈）**，彻底陷入局部最优；
> 2. **课程学习 (Curriculum Learning)**：利用在 10×10 简单棋盘上学会的 `BoardQNet` 自适应池化权重（均分 ~20，最高 34），直接迁移至 20×20；
> 3. **验证步骤**：
>    - **阶段 A (Zero-Shot 验证)**：完全不进行 20×20 训练，直接测试 10×10 权重在 20×20 棋盘上的表现；
>    - **阶段 B (大棋盘微调 Fine-Tuning)**：以 10×10 为先验，调整 $\gamma=0.95$、适度温和探索（$\epsilon=0.3 \to 0.02$），彻底打破 20×20 上的训练壁垒！


In [ ]:
#@title 1. 环境准备 & L4 GPU 验证 & 项目解压
import os, sys, subprocess, pathlib

os.environ.setdefault("SDL_VIDEODRIVER", "dummy")
os.environ["OMP_NUM_THREADS"] = "4"

CONTENT = pathlib.Path("/content")
PROJECT = CONTENT / "snake_rl"
TARBALL = CONTENT / "snake_rl_colab.tar.gz"

if not (PROJECT / "pyproject.toml").exists():
    if not TARBALL.exists():
        print("💡 未在 /content 找到 snake_rl_colab.tar.gz，请上传压缩包：")
        from google.colab import files
        files.upload()
    print("解压项目代码...")
    subprocess.run(["tar", "xzf", str(TARBALL), "-C", str(CONTENT)], check=True)

os.chdir(PROJECT)
print(f"{'cwd':<16}= {os.getcwd()}")
print(f"{'python':<16}= {sys.version.split()[0]}")

# 安装项目（开发模式）
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."], check=True)

import torch
print("\n== GPU 硬件检查 ==") 
cuda_ok = torch.cuda.is_available()
print(f"{'cuda_available':<16}= {cuda_ok}")
if cuda_ok:
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    bf16_ok = torch.cuda.is_bf16_supported()
    print(f"{'gpu_name':<16}= {gpu_name}")
    print(f"{'gpu_vram':<16}= {vram_gb:.2f} GB")
    print(f"{'bfloat16_amp':<16}= {bf16_ok}")
else:
    print("⚠️ 警告: 未检测到 GPU！请在 Colab 运行时中切换至 L4 GPU")


In [ ]:
#@title 2. 准备 10×10 预训练检查点（若不存在则快速预训练 40 万步）
import pathlib, subprocess, sys

ckpt_10 = pathlib.Path("runs/vector_board_10x10_seed0/checkpoint.pt")
if ckpt_10.exists():
    print(f"✅ 检测到已有的 10×10 权重: {ckpt_10}，直接用于迁移！")
else:
    print("⚠️ 未找到已有权重，正在快速进行 10×10 预训练（400,000 步，约 3.5 分钟）...")
    cmd = [
        sys.executable, "scripts/train_vector_dqn.py",
        "--num-envs", "32",
        "--total-steps", "400000",
        "--width", "10",
        "--height", "10",
        "--timeout-factor", "50.0",
        "--timeout-reward", "-10.0",
        "--obs-type", "board",
        "--board-channels", "5",
        "--adaptive-pool-size", "3",
        "--hidden-size", "256",
        "--batch-size", "256",
        "--lr", "1e-3",
        "--gamma", "0.9",
        "--epsilon-start", "1.0",
        "--epsilon-end", "0.05",
        "--epsilon-decay-steps", "100000",
        "--output-dir", "runs/vector_board_10x10_seed0",
        "--device", "cuda",
    ]
    subprocess.run(cmd, check=True)
    print("✅ 10×10 预训练完成！检查点已生成。")


In [ ]:
#@title 3. 阶段 A：Zero-Shot 跨分辨率直接测试（10×10 权重在 20×20 上实测 100 局）
import subprocess, sys

print("=== 正在运行 20×20 棋盘 Zero-Shot 贪心评测（完全未经 20×20 训练） ===")
cmd = [
    sys.executable, "scripts/evaluate.py",
    "--run-dir", "runs/vector_board_10x10_seed0",
    "--width", "20",
    "--height", "20",
    "--timeout-factor", "50.0",
    "--episodes", "100",
    "--seed", "10000",
    "--device", "cuda",
    "--output", "runs/vector_board_10x10_seed0/zero_shot_20x20.json",
]
subprocess.run(cmd, check=True)


In [ ]:
#@title 4. 阶段 B：启动 20×20 课程微调（Fine-Tuning，800,000 步）
import subprocess, sys, time

# 核心超参调整针对 20×20 大棋盘：
# - --gamma 0.95 (视野拓展至 ~20 步，匹配 20×20 更远曼哈顿距离)
# - --epsilon-start 0.3 (保留 70% 预训练知识，温和探索)
# - --lr 5e-4 (温和梯度更新，防止灾难性遗忘)
cmd = [
    sys.executable, "scripts/train_vector_dqn.py",
    "--init-checkpoint", "runs/vector_board_10x10_seed0/checkpoint.pt",
    "--num-envs", "32",
    "--total-steps", "800000",
    "--width", "20",
    "--height", "20",
    "--timeout-factor", "50.0",
    "--timeout-reward", "-10.0",
    "--obs-type", "board",
    "--board-channels", "5",
    "--adaptive-pool-size", "3",
    "--hidden-size", "256",
    "--batch-size", "256",
    "--lr", "5e-4",
    "--gamma", "0.95",
    "--max-memory", "200000",
    "--update-every", "1",
    "--updates-per-step", "2",
    "--epsilon-start", "0.30",
    "--epsilon-end", "0.02",
    "--epsilon-decay-steps", "200000",
    "--target-sync-every", "500",
    "--eval-every-steps", "20000",
    "--eval-episodes", "20",
    "--final-eval-episodes", "100",
    "--device", "cuda",
    "--output-dir", "runs/vector_board_20x20_curriculum_seed0",
]

print("🚀 启动 20×20 课程微调训练...")
t0 = time.perf_counter()
subprocess.run(cmd, check=True)
t_total = time.perf_counter() - t0
print(f"\n20×20 课程微调完成！总耗时: {t_total:.1f}s ({t_total/60:.2f} 分钟)")


In [ ]:
#@title 5. 跨阶段成果横向大对比（E7 从零训练 vs Zero-Shot vs 课程微调）
import json, pathlib

def load_res(p):
    if pathlib.Path(p).exists():
        with open(p) as f: return json.load(f)
    return None

zero_res = load_res("runs/vector_board_10x10_seed0/zero_shot_20x20.json")
curric_res = load_res("runs/vector_board_20x20_curriculum_seed0/final_eval.json")

print("=" * 80)
print("20×20 棋盘大对比：从零训练 vs 课程学习迁移")
print("=" * 80)
print(f"| 实验版本 | 训练步数 | 评测均分 (Mean) | 最高分 (Max) | 撞自身(%) | 撞墙(%) | 超时绕圈(%) |")
print(f"| :--- | :---: | :---: | :---: | :---: | :---: | :---: |
")
print(f"| E7 (20×20 从零训练) | 484,000 | 0.00 | 0 | 0% | 0% | 100% (完全退化绕圈) |")
if zero_res:
    e = zero_res.get("endings", {})
    print(f"| Zero-Shot 零样本迁移 | 0 | **{zero_res['mean_score']:.2f}** | {zero_res['max_score']} | {e.get('self',0)}% | {e.get('wall',0)}% | {e.get('timeout',0)}% |")
if curric_res:
    e = curric_res.get("endings", {})
    print(f"| 课程学习微调 (20×20) | 800,000 | **{curric_res['mean_score']:.2f}** | **{curric_res['max_score']}** | {e.get('self',0)}% | {e.get('wall',0)}% | {e.get('timeout',0)}% |")
print("=" * 80)


In [ ]:
#@title 6. 打包下载 20×20 实验产物
import subprocess
from google.colab import files

archive_name = "runs_20x20_curriculum.tar.gz"
subprocess.run(["tar", "czf", archive_name, "runs/vector_board_20x20_curriculum_seed0"], check=True)
print(f"已打包: {archive_name}")
files.download(archive_name)
